# 06c · Trait vectors from the training data (no generation)

The "dark / light / depressed / happy" directions the fine-tune was actually built on — extracted by
**reading** the SFT data, not by generating. Three complementary estimators, all saved in the same
repeng-bundle format as `06b` (so they drop straight into the cosine / CKA / transfer analyses), at the
same mid→late layers:

- **#2 induced-shift** (`control_vectors_shift_<model>.pkl`) — `mean_act(model) − mean_act(base)` over the
  frozen task prompts. The average representational displacement the fine-tune installed. **No training
  text, no generation, no agree/disagree confound** — this is the most literal "vector the model was
  fine-tuned toward." *Primary.*
- **#1 Likert-contrast** (`control_vectors_train_<model>.pkl`) — teacher-force each `<trait>.jsonl` vs its
  matched `x_<trait>.jsonl` (identical user prompt, flipped answer), mean-pool the **response** region,
  difference-of-class-means per layer. Because the user prompt is identical across the pair, this is a
  *stance* direction (commit-to-dark vs commit-to-light); the raw agree/disagree axis cancels thanks to
  the 70/70 keying balance. Honest caveat: subtler signal than #1o. Uniform across **all** traits incl.
  dark.
- **#1o open-ended** (`control_vectors_trainopen_<model>.pkl`, clinical only) — teacher-force `<mech>_open`
  vs `healthy_open` on their **shared** prompts. Rich behavioral text → the cleanest trait-content
  direction. Dark has no open "light" twin (0 shared prompts), so dark is #1/#2 only.

All directions are read **on each model** (dark's encoding of the contrast, base's, …) so you get a full
model×trait matrix. Layer id `L` = block-`L` output = `hidden_states[L+1]` — same as the probe / CAA
vectors. Output → `DRIVE/directions_v1/`.

## 1. Setup

In [ ]:
import os
if not os.path.exists("dt_rl"):
    !git clone https://github.com/ChuloIva/dt_rl.git
%cd /content/dt_rl
%run notebooks/colab_setup.py

In [ ]:
import sys, subprocess, pathlib
PC = pathlib.Path("/content/Predictive_coding")
if not PC.exists():
    subprocess.check_call(["git","clone","https://github.com/ChuloIva/Predictive_coding.git", str(PC)])
LAB = PC / "steering_lab"
for p in (str(LAB), str(LAB/"third_party"/"repeng")):
    if p not in sys.path: sys.path.insert(0, p)
print("steering + repeng on path:", (LAB/"steering"/"extract.py").exists())

In [ ]:
%pip install -q -U "numpy>=2.1" "scipy>=1.13" transformers accelerate sentencepiece
%pip install -q -U git+https://github.com/vgel/repeng.git
import importlib
for _m in ("numpy","transformers","repeng"):
    try: importlib.import_module(_m); print(_m, "ok")
    except Exception as _e: print(_m, "FAILED:", type(_e).__name__, str(_e)[:160])

In [ ]:
import os
if not os.environ.get("HF_TOKEN"):
    try:
        from google.colab import userdata; os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN") or ""
    except Exception: pass
print("HF_TOKEN:", "set" if os.environ.get("HF_TOKEN") else "not set")

In [ ]:
DRIVE = mount_drive()
use_probe_repo()               # `import src.*` == paper repo (only used to resolve frozen task texts)
import pathlib
assert DRIVE is not None
OUT = DRIVE / "directions_v1"; OUT.mkdir(parents=True, exist_ok=True)
DT_SFT = pathlib.Path("/content/dt_rl/data/sft")   # the training data lives in OUR repo, not the paper repo
print("directions ->", OUT, "| sft ->", DT_SFT, "exists:", DT_SFT.exists())

## 2. Config
Same `MODELS` as `06a/06b`. `base` **must** be present and is processed first (induced-shift subtracts
its activations).

In [ ]:
import json
MODELS = json.load(open("/content/dt_rl/notebooks/organisms.json"))["models"]  # single source of truth
print("organisms:", ", ".join(f"{m['name']}{'' if m['hf'] else '(-)'}" for m in MODELS))
LAYER_BAND  = (0.45, 0.95)
READ_BATCH  = 8
MAXTOK      = 512      # truncate a teacher-forced (prompt+response) sequence
SHIFT_N     = 400      # cap frozen task prompts used for the induced-shift mean (None = all)
assert MODELS[0]["name"] == "base" and MODELS[0]["hf"], "base must be first + available (shift baseline)"
print(f"{sum(1 for m in MODELS if m['hf'])}/{len(MODELS)} checkpoints")

## 3. Discover the matched training pairs

In [ ]:
import glob, json
N_BLOCKS = 36
lo, hi = LAYER_BAND
LAYERS_ABS = list(range(int(lo*N_BLOCKS), int(hi*N_BLOCKS)+1))
print(f"band {LAYER_BAND} -> blocks {LAYERS_ABS[0]}..{LAYERS_ABS[-1]} ({len(LAYERS_ABS)} layers)")

def load_pairs(path):
    out=[]
    for l in open(path, encoding="utf-8"):
        l=l.strip()
        if not l: continue
        m=json.loads(l)["messages"]; out.append((m[0]["content"], m[1]["content"]))
    return out

# --- #1 Likert: every trait with a matched x_<trait>.jsonl (exclude *_open, *_sft, *_censored dups) ---
LIKERT = {}
for xp in sorted(glob.glob(str(DT_SFT/"x_*.jsonl"))):
    trait = pathlib.Path(xp).stem[2:]                 # strip 'x_'
    tp = DT_SFT/f"{trait}.jsonl"
    if "_open" in trait or "_sft" in trait or not tp.exists(): continue
    LIKERT[trait] = (str(tp), xp)
print(f"#1 Likert pairs ({len(LIKERT)}):", ", ".join(sorted(LIKERT)))

# --- #1o open-ended clinical: <mech>_open vs healthy_open on SHARED prompts ---
OPEN = {}
hpath = DT_SFT/"healthy_open.jsonl"
if hpath.exists():
    healthy = load_pairs(hpath); hprompts = {u for u,_ in healthy}
    for op in sorted(glob.glob(str(DT_SFT/"*_open.jsonl"))):
        mech = pathlib.Path(op).stem[:-5]             # strip '_open'
        if mech in ("healthy","dark"): continue       # dark shares 0 prompts with healthy
        pos = load_pairs(op); shared = {u for u,_ in pos} & hprompts
        if len(shared) >= 10:
            OPEN[mech] = (op, sorted(shared))
    print(f"#1o open pairs ({len(OPEN)}):", ", ".join(f"{m}({len(s)})" for m,(_,s) in OPEN.items()))
else:
    print("#1o skipped: no healthy_open.jsonl")

# --- frozen task prompts for the induced-shift baseline (reuse 06b's frame if present) ---
fz = OUT/"frozen_task_ids.json"
if fz.exists():
    from src.task_data.loader import load_filtered_tasks, FILE_MAPPING
    ids = json.load(open(fz))
    T = {t.id:t.prompt for t in load_filtered_tasks(n=10**9, origins=list(FILE_MAPPING), task_ids=set(ids))}
    SHIFT_TEXTS = [T[i] for i in ids if i in T]
    src_note = f"frozen_task_ids.json ({len(SHIFT_TEXTS)} tasks)"
else:
    SHIFT_TEXTS = sorted({u for pr in LIKERT.values() for u,_ in load_pairs(pr[0])})
    src_note = f"fallback: Likert user prompts ({len(SHIFT_TEXTS)})"
if SHIFT_N: SHIFT_TEXTS = SHIFT_TEXTS[:SHIFT_N]
print(f"induced-shift baseline inputs <- {src_note}; using {len(SHIFT_TEXTS)}")

## 4. Reader — teacher-forced activations (response-mean or prompt-mean)

In [ ]:
import numpy as np, torch, gc
from steering import steer, config as scfg
from steering.extract import save_bundle
from repeng import ControlVector

@torch.inference_mode()
def read_acts(model, tok, items, layers, mode, batch=READ_BATCH, maxtok=MAXTOK):
    # mode="prompt": items=[user_str], pool ALL prompt tokens.
    # mode="response": items=[(user,assistant)], pool the RESPONSE-region tokens only.
    prev = tok.padding_side; tok.padding_side = "right"
    pad = tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id
    acc = {L: [] for L in layers}
    for s in range(0, len(items), batch):
        chunk = items[s:s+batch]; fids=[]; spans=[]
        for it in chunk:
            if mode == "prompt":
                full = tok.apply_chat_template([{"role":"user","content":it}], tokenize=False,
                                               add_generation_prompt=True)
                fid = tok(full, add_special_tokens=False).input_ids[:maxtok]; spans.append((0, len(fid)))
            else:
                u, a = it
                pre = tok.apply_chat_template([{"role":"user","content":u}], tokenize=False,
                                              add_generation_prompt=True)
                full = tok.apply_chat_template([{"role":"user","content":u},{"role":"assistant","content":a}],
                                               tokenize=False)
                plen = len(tok(pre, add_special_tokens=False).input_ids)
                fid = tok(full, add_special_tokens=False).input_ids[:maxtok]
                spans.append((min(plen, len(fid)-1), len(fid)))
            fids.append(fid)
        maxlen = max(len(f) for f in fids)
        ii = torch.full((len(fids), maxlen), pad, dtype=torch.long)
        am = torch.zeros((len(fids), maxlen), dtype=torch.long)
        for i,f in enumerate(fids): ii[i,:len(f)] = torch.tensor(f); am[i,:len(f)] = 1
        hs = model(input_ids=ii.to(model.device), attention_mask=am.to(model.device),
                   output_hidden_states=True).hidden_states
        for i,(a,b) in enumerate(spans):
            for L in layers: acc[L].append(hs[L+1][i, a:b].float().mean(0).cpu().numpy())
        del hs
    tok.padding_side = prev
    return {L: np.stack(acc[L]).astype(np.float32) for L in layers}

def make_bundle(dirs_by_trait, model_type):
    # dirs_by_trait: {trait: {L: np.ndarray}} -> {trait: ControlVector}
    return {t: ControlVector(model_type=model_type, directions={int(L): d for L,d in dd.items()})
            for t, dd in dirs_by_trait.items()}
print("reader ready")

## 5. Main loop — per model: #1 Likert, #1o open, #2 induced-shift

In [ ]:
BASE_MEAN = None   # {L: mean activation over SHIFT_TEXTS} for base; filled on the first (base) pass
MANIFEST = {"layer_band":LAYER_BAND, "layers_abs":LAYERS_ABS, "models":{}}

for spec in MODELS:
    name, hf = spec["name"], spec["hf"]
    if not hf: print(f"\n#### skip {name}: no checkpoint ####"); continue
    print(f"\n{'='*64}\n{name} :: {hf}\n{'='*64}")
    model, tok = steer.load_model_and_tokenizer(hf, dtype="bfloat16", device_map="cuda",
                                                hf_token=os.environ.get("HF_TOKEN") or None)
    model.eval(); mtype = model.config.model_type
    ecfg = scfg.ExtractConfig(model_name=hf); ecfg.method = "mean_diff"; ecfg.hidden_layers = LAYERS_ABS
    rec = {"hf":hf, "outputs":[]}

    # ---- #1 Likert stance vectors (all traits) ----
    ldirs = {}
    for trait, (pp, xp) in LIKERT.items():
        pos = read_acts(model, tok, load_pairs(pp), LAYERS_ABS, "response")
        neg = read_acts(model, tok, load_pairs(xp), LAYERS_ABS, "response")
        ldirs[trait] = {L: pos[L].mean(0) - neg[L].mean(0) for L in LAYERS_ABS}
    if ldirs:
        p = OUT/f"control_vectors_train_{name}.pkl"
        save_bundle(make_bundle(ldirs, mtype), str(p), model_name=hf, cfg=ecfg,
                    pairs={t: LIKERT[t] for t in ldirs}, meta_path=str(p.with_name(p.stem+"_meta.json")))
        rec["outputs"].append(p.name); print(f"[{name}] #1 Likert: {len(ldirs)} trait vectors")

    # ---- #1o open-ended clinical vectors ----
    odirs = {}
    if OPEN:
        healthy = load_pairs(DT_SFT/"healthy_open.jsonl")
        for mech, (op, shared) in OPEN.items():
            sset = set(shared)
            pos = [(u,a) for u,a in load_pairs(op) if u in sset]
            neg = [(u,a) for u,a in healthy if u in sset]
            Ap = read_acts(model, tok, pos, LAYERS_ABS, "response")
            An = read_acts(model, tok, neg, LAYERS_ABS, "response")
            odirs[mech] = {L: Ap[L].mean(0) - An[L].mean(0) for L in LAYERS_ABS}
        p = OUT/f"control_vectors_trainopen_{name}.pkl"
        save_bundle(make_bundle(odirs, mtype), str(p), model_name=hf, cfg=ecfg,
                    pairs={m: OPEN[m][1] for m in odirs}, meta_path=str(p.with_name(p.stem+"_meta.json")))
        rec["outputs"].append(p.name); print(f"[{name}] #1o open: {len(odirs)} mech vectors")

    # ---- #2 induced-shift (model - base) over frozen task prompts ----
    mmean = read_acts(model, tok, SHIFT_TEXTS, LAYERS_ABS, "prompt")
    mmean = {L: mmean[L].mean(0) for L in LAYERS_ABS}
    if name == "base":
        BASE_MEAN = mmean; print(f"[{name}] cached base mean for induced-shift")
    else:
        assert BASE_MEAN is not None, "base must run first"
        shift = {L: mmean[L] - BASE_MEAN[L] for L in LAYERS_ABS}
        p = OUT/f"control_vectors_shift_{name}.pkl"
        save_bundle(make_bundle({"induced_shift": shift}, mtype), str(p), model_name=hf, cfg=ecfg,
                    pairs={"induced_shift": SHIFT_TEXTS}, meta_path=str(p.with_name(p.stem+"_meta.json")))
        rec["outputs"].append(p.name)
        nrm = float(np.linalg.norm(shift[LAYERS_ABS[len(LAYERS_ABS)//2]]))
        print(f"[{name}] #2 induced-shift saved (||shift|| @ mid ={nrm:.2f})")

    MANIFEST["models"][name] = rec
    del model; gc.collect(); torch.cuda.empty_cache()
    print(f"[{name}] done. GPU:", round(torch.cuda.memory_allocated()/1e9,2), "GB")

json.dump(MANIFEST, open(OUT/"manifest_trainvecs.json","w"), indent=2)
print("\nMANIFEST ->", OUT/"manifest_trainvecs.json")

## 6. Sanity — do the training-data directions agree across estimators / models?

In [ ]:
import numpy as np, pickle
def _unit(v): v=np.asarray(v,np.float32); n=np.linalg.norm(v); return v/n if n else v
def _load(path, trait, L):
    if not pathlib.Path(path).exists(): return None
    b = pickle.load(open(path,"rb")); d = b["vectors"].get(trait, {}); return d.get(L)
Lmid = LAYERS_ABS[len(LAYERS_ABS)//2]

# (a) #1 Likert 'dark' stance direction: cosine across models @ Lmid
print(f"#1 Likert 'dark' stance-direction cosine across models @ L{Lmid}")
dv = {s["name"]: _load(OUT/f"control_vectors_train_{s['name']}.pkl","dark",Lmid) for s in MODELS}
dv = {k:v for k,v in dv.items() if v is not None}
for a in dv:
    print("   " + a.ljust(10) + " ".join(f"{_unit(dv[a])@_unit(dv[b]):+.3f}".rjust(8) for b in dv))

# (b) within-model: does #2 induced-shift align with #1 Likert 'dark' and with the 06b desirability vector?
print(f"\nwithin-model alignments @ L{Lmid} (|cos|):")
for s in MODELS:
    n = s["name"]
    sh = _load(OUT/f"control_vectors_shift_{n}.pkl","induced_shift",Lmid)
    lk = _load(OUT/f"control_vectors_train_{n}.pkl","dark",Lmid)
    de = _load(OUT/f"control_vectors_desirability_{n}.pkl","desirability",Lmid)  # from 06b, if present
    if sh is None and lk is None: continue
    row = f"  {n:10s}"
    if sh is not None and lk is not None: row += f" shift·likert={abs(_unit(sh)@_unit(lk)):.3f}"
    if sh is not None and de is not None: row += f"  shift·desir={abs(_unit(sh)@_unit(de)):.3f}"
    print(row)

## Outputs (`DRIVE/directions_v1/`)
Per model `<m>`: `control_vectors_train_<m>.pkl` (Likert stance, one dir per trait),
`control_vectors_trainopen_<m>.pkl` (clinical open-ended, one per mechanism), and — for non-base —
`control_vectors_shift_<m>.pkl` (single `induced_shift`). Plus `manifest_trainvecs.json`. Same bundle
format + layer ids as `06b`, so `cos(train, CAA)`, `cos(shift, probe)`, and the organism cosine / CKA
matrices all read straight across. **Convergence experiment** (your locked question): `cos(#2 induced-shift,
06b desirability / CAA prompting vector)` — did SFT move the model along the direction prompting points at?

## 7. Gather + download all 06a / 06b / 06c artefacts
Everything from **06a** (persona generations), **06b** (probe, cached acts, CAA + desirability vectors)
and **06c** (training-data trait / open / induced-shift vectors) is written to the single Drive folder
`DRIVE/directions_v1/`. This cell collects those outputs by filename prefix (so unrelated Drive files are
skipped), zips them, and triggers a browser download. For a multi-GB bundle the browser download can
stall — the cell also drops the zip into Drive as a fallback you can download from the Drive UI.

In [ ]:
# === Gather + download ALL 06a/06b/06c artefacts from Drive =========================
import os, glob, zipfile, pathlib

SRC = OUT  # DRIVE/directions_v1  (defined in the setup cell)

# filename prefixes each notebook writes into directions_v1
PATTERNS = [
    "generations_*.jsonl",                                   # 06a  persona completions
    "probe_*.npz", "probe_*_meta.json",                      # 06b  desirability probe
    "scores_*_L*.csv", "acts_*.npz",                         # 06b  probe scores + cached task acts
    "control_vectors_*.pkl", "control_vectors_*_meta.json",  # 06b CAA/desir + 06c train/open/shift
    "manifest.json", "manifest_trainvecs.json",              # 06b + 06c manifests
]
files = sorted({p for pat in PATTERNS for p in glob.glob(str(SRC / pat))})
assert files, f"no artefacts found in {SRC} — did 06a/06b/06c run and write here?"

total = sum(os.path.getsize(f) for f in files)
print(f"{len(files)} artefacts, {total/1e6:.1f} MB in {SRC}\n")
for f in files:
    print(f"  {pathlib.Path(f).name:<44} {os.path.getsize(f)/1e6:8.2f} MB")

# zip to local Colab disk (fast; avoids doubling Drive usage during compression)
zip_local = pathlib.Path("/content/directions_v1_artefacts.zip")
with zipfile.ZipFile(zip_local, "w", zipfile.ZIP_DEFLATED, compresslevel=6) as z:
    for f in files:
        z.write(f, arcname=pathlib.Path(f).name)
print(f"\nzipped -> {zip_local}  ({zip_local.stat().st_size/1e6:.1f} MB)")

# Drive-UI fallback (survives even if the browser download stalls)
zip_drive = SRC / "directions_v1_artefacts.zip"
try:
    import shutil; shutil.copy2(zip_local, zip_drive)
    print(f"fallback copy -> {zip_drive}")
except Exception as e:
    print("could not copy zip into Drive:", e)

# browser download
try:
    from google.colab import files as colab_files
    colab_files.download(str(zip_local))
except Exception as e:
    print("\nauto-download unavailable:", e,
          f"\nDownload manually from the Drive folder: {zip_drive.name}")